#### Nodes
* A Node is a simply a function that performs one task inside a langGraph.
* Think of it like one node = one task
* in python we write a function , in langgraph this function becoms a node.
    ``` python
    def greet():
        return "Hello"
    ```
##### Example

``` python
def greet_node(state):
    print("Hello")
    return state
```
* This is a node it recives state it performs work ``hello`` and returns the ``state``

``` markdown
Recieves state
        ↓
    Process
        ↓
    return updated state
```

``` markdown
           State

             │

             ▼

       Node Function

             │

     Do Some Work

             │

             ▼

      Updated State
```

In [1]:
## Example
state = {"name":"subbu"}

def welocome_node(state):
    print("welcome",state['name'])
    return state

call = welocome_node(state)
print(call)

welcome subbu
{'name': 'subbu'}


Node Functions

In [3]:
# every function is a node
# example 1: 

state = {"number":5}
def square_node(state):
    number = state["number"]
    square = number ** 2
    return {
        "Square" : square
    }

call  = square_node(state)
print(call)

{'Square': 25}


In [4]:
# example 2
def upper_case_node(state):
    text = state["text"]
    return {
        "text": text.upper()
    }

call = upper_case_node(state={"text":"subramani"})
print(call)


{'text': 'SUBRAMANI'}


Reading a State

In [5]:
# state is like a dictionary
state = {
"name":"Subbu",
"age":25
}

state["name"]

'Subbu'

Updating a State

In [7]:
state = {"name":"subbu"}

def add_age(state):
    state["age"] = 25
    return state

call = add_age(state)
print(call)

{'name': 'subbu', 'age': 25}


In [12]:
# Example
state = {"Marks" : 70}

def calculate_grade(state):
    if state["Marks"] >= 70:
        state["grade"] = "A"
    return state

call = calculate_grade(state)
print(call)


{'Marks': 70, 'grade': 'A'}


Stateless Nodes

A stateless node derives everything it needs from the state passed in. Given the same input state, it will always produce the same output. No database calls, no external memory.

This node is purely functional. You can test it in isolation, replay it, and it behaves predictably every time.


In [ ]:
def calculate_tax_node(state: InvoiceState) -> dict:
    amount = state["invoice_amount"]
    tax_rate = state["tax_rate"]
    tax = round(amount * tax_rate, 2)
    return {"tax_amount": tax}

Stateful Nodes

A stateful node reaches outside the graph to read from or write to persistent external storage — databases, caches, files, external APIs with side effects.


In [ ]:
import sqlite3

def load_user_profile_node(state: UserState) -> dict:
    # Reaching into an external database — this is stateful
    conn = sqlite3.connect("users.db")
    cursor = conn.cursor()
    cursor.execute("SELECT name, tier FROM users WHERE id = ?", (state["user_id"],))
    row = cursor.fetchone()
    conn.close()

    if row:
        return {"user_name": row[0], "user_tier": row[1]}
    return {"user_name": "Unknown", "user_tier": "free"}

| Stateless Node             | Stateful Node                          |
| -------------------------- | -------------------------------------- |
| Doesn't modify state       | Reads and updates state                |
| Same output for same input | Output can depend on accumulated state |
| Easy to test               | Useful for workflows that need memory  |
| Example: formatting text   | Example: tracking conversation history |


Reusable Nodes

A reusable node is a node function designed to be registered multiple times in the same graph under different names, or used across different graphs entirely.

Pattern 1 — Same function, multiple names in one graph:


In [ ]:
from typing import TypedDict

class DocState(TypedDict):
    raw_text: str
    step1_output: str
    step2_output: str

def normalize_text(state: DocState, target_field: str, source_field: str) -> dict:
    text = state[source_field]
    normalized = text.strip().lower().replace("\n", " ")
    return {target_field: normalized}

# You cannot pass extra args directly, so wrap in a closure
def make_normalizer(source_field: str, target_field: str):
    def node(state: DocState) -> dict:
        text = state[source_field]
        normalized = text.strip().lower().replace("\n", " ")
        return {target_field: normalized}
    return node

builder = StateGraph(DocState)

# Same logic, two different node registrations
builder.add_node("normalize_raw", make_normalizer("raw_text", "step1_output"))
builder.add_node("normalize_step1", make_normalizer("step1_output", "step2_output"))

In [ ]:
#Pattern 2 — Class-based reusable nodes across graphs:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

class LLMCallNode:
    def __init__(self, system_prompt: str, output_field: str):
        self.system_prompt = system_prompt
        self.output_field = output_field
        self.llm = ChatOpenAI(model="gpt-4o-mini")

    def __call__(self, state: dict) -> dict:
        user_text = state.get("user_input", "")
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": user_text}
        ]
        response = self.llm.invoke(messages)
        return {self.output_field: response.content}

# Reused across different graphs with different configs
summarizer = LLMCallNode(
    system_prompt="Summarize the following text concisely.",
    output_field="summary"
)

classifier = LLMCallNode(
    system_prompt="Classify the sentiment as positive, negative, or neutral.",
    output_field="sentiment"
)

# Graph 1
builder1 = StateGraph(SummaryState)
builder1.add_node("summarize", summarizer)

# Graph 2
builder2 = StateGraph(SentimentState)
builder2.add_node("classify", classifier)

In [ ]:
# Pattern 3 — Utility nodes shared across projects as a module:
# nodes/shared.py  — your reusable node library

def log_state_node(state: dict) -> dict:
    """A debug node that prints state and returns nothing."""
    print("=== STATE SNAPSHOT ===")
    for key, value in state.items():
        print(f"  {key}: {value}")
    return {}

def reset_counter_node(counter_field: str):
    """Factory: returns a node that resets a named counter field to zero."""
    def node(state: dict) -> dict:
        return {counter_field: 0}
    return node

In [ ]:
## Complete flow Example
state = {
    "name": "Subbu",
    "marks": 92
}

# Node 1: reads the name

def reading_node(state):
    print(state["name"])
    return state

call = reading_node(state) # after execution no chnages in the state becuase it is only reading and returning.
print(call)

# calculkate grade 
def calculate_grade(state):
    if state["marks"] >= 90:
        state["grade"] = "A+"
    elif state["marks"] >= 75:
        state["grade"] = "A"
    else:
        state["grade"] = "B"
    return state
call = calculate_grade(state) # here the state is getting updating with the grades.
print(call)


# generate message
def congratulate(state):
    state["message"] = (
        f"Congratulations {state['name']}! "
        f"You received grade {state['grade']}."
    )
    return state

call = congratulate(state) # Each node performs one clear responsibility, and together they form a complete workflow.
print(call)

Subbu
{'name': 'Subbu', 'marks': 92}
{'name': 'Subbu', 'marks': 92, 'grade': 'A+'}
{'name': 'Subbu', 'marks': 92, 'grade': 'A+', 'message': 'Congratulations Subbu! You received grade A+.'}


Key Takeaways

* A Node is a Python function that represents one step in a LangGraph workflow.
* Every node receives the current state, performs a task, and returns an updated state.
* Nodes can read values from the state, add new values, or modify existing ones.
* Returning the updated state is essential because subsequent nodes depend on it.
* Stateless nodes don't change or rely on accumulated state, while stateful nodes use and update shared state across the workflow.
* Well-designed nodes are small, focused, and reusable, making graphs easier to maintain and extend.

In [24]:
# code understadning

from langgraph.graph import StateGraph
from typing import TypedDict

class MyState(TypedDict):
    message:str

def my_node(state:MyState):
    print("=====Node is running====")
    return {"message": "Hello from node"}
    
bulider = StateGraph(MyState)
bulider.add_node("my_node",my_node) # here registering the node to the graph


call = my_node(state)
print(call)

=====Node is running====
{'message': 'Hello from node'}


Node Functions

A node function has a strict signature that LangGraph expects:

``` markdown
(state) -> dict   OR   (state, config) -> dict
```

The first argument is always the current state. The second optional argument is a RunnableConfig object that carries runtime metadata like thread IDs, tags, and callbacks.


In [29]:
from langchain_core.runnables import RunnableConfig
from typing import TypedDict

class AgentState(TypedDict):
    user_query: str
    response: str
    attempt_count: int

# Minimal form — only state
def simple_node(state: AgentState) -> dict:
    return {"response": "done"}

# Full form — state + config
def node_with_config(state: AgentState, config: RunnableConfig) -> dict:
    thread_id = config["configurable"].get("thread_id", "unknown")
    print(f"Running in thread: {thread_id}")
    return {"response": f"Answered for thread {thread_id}"}

call1 = simple_node(state)
print(call1)
# Define state and config here
state = {"user_query": "hello", "response": "", "attempt_count": 0}
config = {"configurable": {"thread_id": "123"}}
call2 = node_with_config(state,config)
print(call2)


{'response': 'done'}
Running in thread: 123
{'response': 'Answered for thread 123'}
